# 04 LDMOS output-network screening

Simple network sweeps. This is not a nonlinear PA model; it is a way to screen candidate load networks and current/voltage stress.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from rot_eo_model.constants import DEFAULTS
from rot_eo_model.rf_lumped import z_capacitor, shunt_parallel, series, voltage_divider_load_voltage, cap_current_peak_A

In [ ]:
freqs = np.linspace(6e6, 102e6, 1000)
w = 2*np.pi*freqs
C = DEFAULTS.capacitance_F
Zc = z_capacitor(freqs, C)
Vsrc = 120.0  # placeholder Thevenin source peak per side/differential equivalent
Zsrc = 25.0  # placeholder source/output resistance

def network_voltage(params):
    # params: series L [nH], series R [ohm], shunt R [ohm]
    L_nH, Rs, Rsh = params
    ZL = 1j*w*(L_nH*1e-9)
    Zload = shunt_parallel(Rsh, Zc)
    Ztot_load = series(Rs, ZL, Zload)
    return voltage_divider_load_voltage(Vsrc, Zsrc, Ztot_load) * (Zload / Ztot_load)  # voltage across EO/shunt node

def objective(params):
    V = network_voltage(params)
    mag = np.abs(V)
    # maximize minimum and punish huge peaking; return scalar to minimize
    min_mag = np.min(mag)
    ripple = np.std(20*np.log10(mag/np.maximum(np.mean(mag), 1e-12)))
    return -min_mag + 0.2*ripple

bounds = [(0, 300), (0, 20), (300, 5000)]
res = differential_evolution(objective, bounds, seed=1, polish=False, maxiter=40)
res.x, res.fun

In [ ]:
params = res.x
V = network_voltage(params)
plt.figure()
plt.plot(freqs/1e6, np.abs(V))
plt.xlabel("Frequency [MHz]")
plt.ylabel("EO node voltage [Vpeak, arbitrary source]")
plt.title(f"Toy optimized low-Q network: L={params[0]:.1f} nH, Rs={params[1]:.1f} Ω, Rsh={params[2]:.0f} Ω")
plt.grid(True)

plt.figure()
plt.plot(freqs/1e6, np.unwrap(np.angle(V)))
plt.xlabel("Frequency [MHz]")
plt.ylabel("Phase [rad]")
plt.title("Transfer phase")
plt.grid(True)

TODO: replace this toy source model with a transistor/load-line-inspired output model and/or scikit-rf networks from measured S-parameters.